# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — "What Predicts Health?" (Random Forest feature importance, ML Appendix).** The paper reports Average Position (43%) and Impressions (32%) as the top predictors of Health Score, and is careful to flag this itself: Health Score is defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) so two of the three top "predictors" are literally addends in the label's own formula.

**My methodology question:** this is the label-derived-features pattern from the leakage taxonomy the label was computed FROM these columns, and the columns are also in the features. The paper's own caveat ("importance is descriptive rather than causal") is the right instinct, but the taxonomy gives a concrete next step: re-run the Random Forest once *without* Position and Impressions, and see whether importance collapses onto Scroll Depth and CTR the way the leakage skill predicts (a ~1.0 → ~0.7-style collapse is the confession). Without that train-without test, I can't tell whether Scroll Depth (15%) and CTR (8%) carry any real signal of their own, or whether they only look small because Position/Impressions are soaking up label-derived variance first.

**Finding B  "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, ML Appendix).** Finding #1 earlier in the same paper reports 74.8K growing vs 45.6K declining pages in the portfolio — a majority class of about 62%. The methodology page says this model used an 80/20 split with no mention of grouping by client or brand.

**My methodology question, in two parts:**
1. **Base rate:** if the 61.8K-row ML sample has a similar ~62% majority class, 71% accuracy is only about 9 points of real skill over always guessing "growing" not a headline number on its own. The paper doesn't print the holdout base rate next to the 71%, so a reader can't check this.
2. **Split honesty:** the portfolio spans 57 brands. A random 80/20 row split lets pages from the same brand land in both train and test, so the model can partly memorize brand-level quirks (a brand's typical word count, typical position) rather than learning a signal that transfers to a brand it hasn't seen. Section 2 below runs exactly this comparison on my own model, and the gap is real so it's a fair question to ask of the paper's number too, not a hypothetical one.

In [3]:
# Sanity check: reproduce the base-rate math behind my methodology question on Finding B,
# using the paper's own reported counts (Finding #1: 74.8K growing vs 45.6K declining).
growing, declining = 74_800, 45_600
majority_share = max(growing, declining) / (growing + declining)
reported_accuracy = 0.71

print("portfolio-wide majority class share:", round(majority_share, 3))
print("reported holdout accuracy:", reported_accuracy)
print("skill above naive majority-class guess:", round(reported_accuracy - majority_share, 3))

portfolio-wide majority class share: 0.621
reported holdout accuracy: 0.71
skill above naive majority-class guess: 0.089


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model (`w05_model_(1).ipynb`) already used `GroupShuffleSplit` on `client_id` from the start, so there's no committed "before" number to point to. To actually show the before/after honesty check this assignment asks for — and to answer my own methodology question on Finding B above I rebuild the exact same Logistic Regression here twice, same features, same seed, same model, and change only the split:

- **Before:** naive random row-level split (`ShuffleSplit`) — the kind of split the paper's growth model appears to use.
- **After:** grouped split by `client_id` (`GroupShuffleSplit`) — every client entirely in train or test, never both, matching Week-5.

In [4]:
!git clone https://github.com/HasnainRaza16/flyrank-ml-Hasnain.git
%cd flyrank-ml-Hasnain

Cloning into 'flyrank-ml-Hasnain'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 173 (delta 73), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 1.90 MiB | 6.84 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/flyrank-ml-Hasnain/flyrank-ml-Hasnain


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- Same safe feature set as Week-5: current-window, non-label, non-ID ---
numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]

df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
numeric_features += ["has_keyword_data", "has_word_count"]

df[numeric_features] = df[numeric_features].fillna(0)
df[categorical_features] = df[categorical_features].fillna("unknown")

X = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
y = df["is_declining_label"]
groups = df["client_id"]

print("rows, cols:", df.shape, "| n clients:", groups.nunique(), "| base rate:", round(y.mean(), 3))

rows, cols: (30000, 47) | n clients: 32 | base rate: 0.542


In [6]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def fit_and_score(train_idx, test_idx, label):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    logreg = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
    logreg.fit(X_train_s, y_train)
    proba = logreg.predict_proba(X_test_s)[:, 1]

    overlap = len(set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"]))
    return {
        "split": label,
        "roc_auc": round(roc_auc_score(y_test, proba), 3),
        "precision_at_50": round(precision_at_k(proba, y_test.values, 50), 3),
        "precision_at_100": round(precision_at_k(proba, y_test.values, 100), 3),
        "test_base_rate": round(y_test.mean(), 3),
        "test_clients": df.iloc[test_idx]["client_id"].nunique(),
        "train_test_client_overlap": overlap,
    }

# --- BEFORE: naive random row-level split ---
ss = ShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx_r, test_idx_r = next(ss.split(X, y))
result_random = fit_and_score(train_idx_r, test_idx_r, "random (before)")

# --- AFTER: grouped split by client_id, matching Week-5 ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx_g, test_idx_g = next(gss.split(X, y, groups=groups))
result_grouped = fit_and_score(train_idx_g, test_idx_g, "grouped by client (after)")

comparison = pd.DataFrame([result_random, result_grouped])
print("=== BEFORE/AFTER: RANDOM SPLIT vs GROUPED (CLIENT-HELD-OUT) SPLIT ===")
print(comparison.to_string(index=False))
print()
print("ROC AUC gap (random − grouped):", round(result_random["roc_auc"] - result_grouped["roc_auc"], 3))

=== BEFORE/AFTER: RANDOM SPLIT vs GROUPED (CLIENT-HELD-OUT) SPLIT ===
                    split  roc_auc  precision_at_50  precision_at_100  test_base_rate  test_clients  train_test_client_overlap
          random (before)    0.693             0.88              0.88           0.546            32                         32
grouped by client (after)    0.589             0.80              0.74           0.517             8                          0

ROC AUC gap (random − grouped): 0.104


**Reading the gap.** The random split shows every one of the 32 clients on both sides of the train/test boundary (`train_test_client_overlap: 32`) the model gets to see each client's style during training and is then tested on more rows from those same clients. The grouped split holds out 8 clients entirely (`overlap: 0`), so the test question changes from "does this look like a client I've trained on?" to "does this hold up on a client I've never seen?" That honest question costs about 0.10 ROC AUC and a real drop in precision@50 (0.88 → 0.80). This is the same shape of gap my methodology question raised about the paper's growth model in Section 1, a random split with 57 brands in play very plausibly overstates how the model will do on brand 58. My own grouped number, 0.589 AUC on a 0.517 base rate, is the one I'd actually stand behind; the 0.693 random number is the one I'd be tempted to lead with, and it's the less honest one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist against the feature set used above:

- **Label-derived / sibling columns:** `trend_direction` and `trend_pct` (the label source) are excluded confirmed, not in `numeric_features` or `categorical_features`.
- **Decision-derived / product flags:** no `*_tier` bucket, `provider_used`, or `model_used` column is used as a feature confirmed.
- **IDs as features:** `content_id` / `client_id` are used only for grouping, never one-hot encoded into `X` confirmed.
- **Future/overlapping windows — the one worth actually testing:** the label comes from `impressions_last_30d` vs `impressions_prev_30d`. My features include `impressions_90d`, `clicks_90d`, `sessions_90d`, and five similar 90-day totals. The data dictionary's own leakage warning for the full warehouse release is about exactly this shape of overlap, so I check whether it also applies to this 30k-row teaching slice.

In [7]:
# Step 1 — does the 90-day feature window arithmetically CONTAIN the label's 30-day windows?
check = df.copy()
check["impressions_prev_30d"] = check["impressions_prev_30d"].fillna(0)
check["earlier_30d"] = check["impressions_90d"] - check["impressions_last_30d"] - check["impressions_prev_30d"]

print("rows where impressions_90d < last_30d + prev_30d (should be 0):", (check["earlier_30d"] < 0).sum())
print("correlation, impressions_90d vs impressions_last_30d:", round(df["impressions_90d"].corr(df["impressions_last_30d"]), 3))

rows where impressions_90d < last_30d + prev_30d (should be 0): 0
correlation, impressions_90d vs impressions_last_30d: 0.918


In [8]:
# Step 2 — the actual test the skill calls for: train WITH the suspect window totals, then WITHOUT, on the honest (grouped) split.
# A collapse toward the base rate would be the confession that the overlap is being exploited.
suspects = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
            "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d"]
numeric_without_suspects = [f for f in numeric_features if f not in suspects]

def build_X(numeric_feats):
    return pd.get_dummies(df[numeric_feats + categorical_features], columns=categorical_features, drop_first=True)

def score_feature_set(numeric_feats, label):
    Xf = build_X(numeric_feats)
    train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED).split(Xf, y, groups=groups))
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xf.iloc[train_idx])
    Xte = scaler.transform(Xf.iloc[test_idx])
    lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
    lr.fit(Xtr, y.iloc[train_idx])
    auc = roc_auc_score(y.iloc[test_idx], lr.predict_proba(Xte)[:, 1])
    print(f"{label}: n_features={Xf.shape[1]:>3}  ROC AUC={auc:.3f}")
    return auc

auc_with = score_feature_set(numeric_features, "WITH 90d-window totals (suspects)")
auc_without = score_feature_set(numeric_without_suspects, "WITHOUT 90d-window totals")
print("collapse:", round(auc_with - auc_without, 3), "(near-zero = suspects aren't doing leaky work)")

WITH 90d-window totals (suspects): n_features= 33  ROC AUC=0.589
WITHOUT 90d-window totals: n_features= 25  ROC AUC=0.588
collapse: 0.001 (near-zero = suspects aren't doing leaky work)


**Verdict: flagged, investigated, not confirmed.** The 90 day totals do arithmetically contain the label's 30-day windows (`earlier_30d` is never negative, and `impressions_90d` correlates 0.92 with `impressions_last_30d`) on paper this matches the future/overlapping-window pattern in the taxonomy. But the train-without test is what actually decides it, and removing the eight suspect columns changes ROC AUC by about 0.001-no collapse. The rate and consistency features (`ctr`, `avg_position`, `days_with_impressions`, `days_with_sessions`, `engagement_rate`) are carrying the real signal; the raw 90-day totals were correlated with the label window but not load-bearing for the model. I'm keeping them, but I'm noting this rather than assuming "no leakage" from the checklist alone  this is the "too good, investigated, not celebrated" step for a case that turned out fine, which is itself worth showing rather than only showing the cases that fail.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest phrasing I've come close to using.** My generated `outputs/model_report.md` labels queue items with reason codes like `model_decline_risk` and `declining_with_demand`, and it would be an easy, natural slide from there into a summary line like:

> "The model predicts which pages will decline."

That's the sentence I'd actually be tempted to write in a stakeholder summary, and it overclaims in two ways this notebook now has receipts for: it implies a forecast about the *future*, and it implies the number holds up in general, when Section 2 shows the honest, client-held-out number (0.589 AUC) is meaningfully weaker than the number a random split would hand me (0.693 AUC).

**Rewrite:**

> "On pages from clients the model has not seen during training, the model's ranking is directionally better than random at surfacing pages with an *already-observed* decline in impressions (ROC AUC 0.589 vs a 0.517 base rate, client-held-out split). It is decision-support for prioritizing which existing pages a reviewer opens first ,it does not forecast future decline, and its lift over guessing is modest, not dramatic."

This keeps the words the assignment asks for *observed* (the label is a measured past outcome, not a forecast), *measured* (0.589 AUC against a stated 0.517 base rate, not a bare adjective), *directional* (better than random, not "accurate"), and *decision-support* (helps a reviewer prioritize, doesn't replace their judgment) — and it's the version I'd actually be able to defend if someone asked me to show the split.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.